# Minesweeper with a Deep Q-Network (DQN)

In [ ]:
import itertools
import random
from collections import deque
from dataclasses import dataclass
from html import escape

import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import HTML, Markdown, display
from torch import nn

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'mps'
    if torch.backends.mps.is_available()
    else 'cpu'
)

print('Torch version:', torch.__version__)
print('Using device:', DEVICE)


## 1. Core Game Logic

The `MineBoard` class stores the hidden board, places mines, counts neighboring mines, and reveals cells recursively when the player clicks an empty square.


In [ ]:
class MineBoard:
    MINE = '@'
    UNCOVERED = ' '
    HIDDEN = '?'

    def __init__(self, size=5, num_mines=10, rng=None):
        if size <= 0:
            raise ValueError('size must be positive')
        if num_mines < 0 or num_mines >= size * size:
            raise ValueError('num_mines must be between 0 and size * size - 1')
        self._board = [[self.HIDDEN] * size for _ in range(size)]
        self._size = size
        self._num_mines = num_mines
        self._rng = rng or random
        mine_positions = self._rng.sample(
            list(itertools.product(range(size), range(size))), num_mines
        )
        for i, j in mine_positions:
            self._board[i][j] = self.MINE
        self._neighbouring_mines = [[0] * size for _ in range(size)]
        for i in range(size):
            for j in range(size):
                self._neighbouring_mines[i][j] = self._count_neighbouring_mines(i, j)

    def print(self):
        for row in self.board():
            print(row)

    def board(self):
        retval = []
        for i in range(len(self._board)):
            new_row = []
            for j in range(len(self._board[i])):
                if self._board[i][j] == self.MINE or self._board[i][j] == self.HIDDEN:
                    new_row.append(self.HIDDEN)
                else:
                    if self._neighbouring_mines[i][j] == 0:
                        new_row.append(self.UNCOVERED)
                    else:
                        new_row.append(str(self._neighbouring_mines[i][j]))
            retval.append(new_row)
        return retval

    def size(self):
        return self._size

    def num_mines(self):
        return self._num_mines

    def safe_cells(self):
        return self._size * self._size - self._num_mines

    def revealed_safe_cells(self):
        count = 0
        for i in range(len(self._board)):
            for j in range(len(self._board[i])):
                if self._board[i][j] == self.UNCOVERED:
                    count += 1
        return count

    def has_mine(self, i, j):
        return self._board[i][j] == self.MINE

    def is_hidden(self, i, j):
        return self._board[i][j] == self.HIDDEN or self._board[i][j] == self.MINE

    def visible_value(self, i, j):
        if self.is_hidden(i, j):
            return self.HIDDEN
        if self._neighbouring_mines[i][j] == 0:
            return self.UNCOVERED
        return str(self._neighbouring_mines[i][j])

    def perform_action(self, i, j):
        if self._board[i][j] == self.MINE:
            return False
        if self._board[i][j] == self.HIDDEN:
            self._board[i][j] = self.UNCOVERED
            if self._neighbouring_mines[i][j] == 0:
                for neighbour_i, neighbour_j in self._neighbours(i, j):
                    self.perform_action(neighbour_i, neighbour_j)
        return True

    def _count_neighbouring_mines(self, i, j):
        count = 0
        for neighbour_i, neighbour_j in self._neighbours(i, j):
            if self._board[neighbour_i][neighbour_j] == self.MINE:
                count += 1
        return count

    def _neighbours(self, i, j):
        retval = []
        for neighbour_i in range(max(0, i - 1), min(len(self._board), i + 2)):
            for neighbour_j in range(max(0, j - 1), min(len(self._board[i]), j + 2)):
                if neighbour_i != i or neighbour_j != j:
                    retval.append((neighbour_i, neighbour_j))
        return retval

    def is_solved(self):
        for i in range(len(self._board)):
            for j in range(len(self._board[i])):
                if self._board[i][j] == self.HIDDEN:
                    return False
        return True


## 2. Turn the Game into an RL Environment

A DQN agent needs four things from the game:

- a numeric **state**
- a discrete **action** space
- a **reward** after each action
- a **done** signal when the episode ends

The environment below uses a one-hot encoding for each cell:

- hidden
- empty uncovered cell
- numbers 1 through 8


In [ ]:
class MinesweeperEnv:
    HIDDEN_CHANNEL = 0
    EMPTY_CHANNEL = 1
    NUMBER_OFFSET = 2
    CHANNELS_PER_CELL = 10

    def __init__(
        self,
        size=5,
        num_mines=10,
        reveal_reward=0.1,
        win_reward=1.0,
        lose_reward=-1.0,
        invalid_reward=-0.25,
        safe_first_move=False,
        seed=None,
    ):
        self._size = size
        self._num_mines = num_mines
        self._reveal_reward = reveal_reward
        self._win_reward = win_reward
        self._lose_reward = lose_reward
        self._invalid_reward = invalid_reward
        self._safe_first_move = safe_first_move
        self._rng = random.Random(seed)
        self._board = None
        self._moves_taken = 0

    def reset(self, seed=None):
        if seed is not None:
            self._rng.seed(seed)
        self._board = MineBoard(self._size, self._num_mines, rng=self._rng)
        self._moves_taken = 0
        return self.state()

    def size(self):
        return self._size

    def action_space_size(self):
        return self._size * self._size

    def available_actions(self):
        if self._board is None:
            raise RuntimeError('reset() must be called before available_actions()')
        actions = []
        for i in range(self._size):
            for j in range(self._size):
                if self._board.is_hidden(i, j):
                    actions.append(i * self._size + j)
        return actions

    def state(self):
        if self._board is None:
            raise RuntimeError('reset() must be called before state()')
        encoded = []
        for i in range(self._size):
            for j in range(self._size):
                channels = [0.0] * self.CHANNELS_PER_CELL
                visible_value = self._board.visible_value(i, j)
                if visible_value == MineBoard.HIDDEN:
                    channels[self.HIDDEN_CHANNEL] = 1.0
                elif visible_value == MineBoard.UNCOVERED:
                    channels[self.EMPTY_CHANNEL] = 1.0
                else:
                    number = int(visible_value)
                    channels[self.NUMBER_OFFSET + number] = 1.0
                encoded.extend(channels)
        return encoded

    def step(self, action):
        if self._board is None:
            raise RuntimeError('reset() must be called before step()')
        if action < 0 or action >= self.action_space_size():
            raise ValueError('action is out of range')

        row, col = divmod(action, self._size)
        if self._safe_first_move and self._moves_taken == 0 and self._board.has_mine(row, col):
            while self._board.has_mine(row, col):
                self._board = MineBoard(self._size, self._num_mines, rng=self._rng)

        if not self._board.is_hidden(row, col):
            return self.state(), self._invalid_reward, False, {
                'invalid_action': True,
                'row': row,
                'col': col,
                'revealed': 0,
            }

        revealed_before = self._board.revealed_safe_cells()
        survived = self._board.perform_action(row, col)
        self._moves_taken += 1

        if not survived:
            return self.state(), self._lose_reward, True, {
                'hit_mine': True,
                'row': row,
                'col': col,
                'revealed': 0,
            }

        revealed = self._board.revealed_safe_cells() - revealed_before
        reward = revealed * self._reveal_reward
        done = self._board.is_solved()
        if done:
            reward += self._win_reward

        return self.state(), reward, done, {
            'hit_mine': False,
            'invalid_action': False,
            'row': row,
            'col': col,
            'revealed': revealed,
        }

    def render(self):
        if self._board is None:
            raise RuntimeError('reset() must be called before render()')
        return self._board.board()


## 3. Build the Deep Q-Network Agent

The DQN stores experiences in a replay buffer, predicts Q-values with a neural network, and periodically copies its learned weights into a target network for more stable training.


In [ ]:
@dataclass
class Experience:
    state: list[float]
    action: int
    reward: float
    next_state: list[float]
    done: bool
    next_actions: list[int]


class ReplayBuffer:
    def __init__(self, capacity):
        self._buffer = deque(maxlen=capacity)

    def add(self, experience):
        self._buffer.append(experience)

    def sample(self, batch_size, rng):
        return rng.sample(list(self._buffer), batch_size)

    def __len__(self):
        return len(self._buffer)


class QNetwork(nn.Module):
    def __init__(self, input_size, output_size, hidden_sizes=(128, 64)):
        super().__init__()
        hidden_one, hidden_two = hidden_sizes
        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_one),
            nn.ReLU(),
            nn.Linear(hidden_one, hidden_two),
            nn.ReLU(),
            nn.Linear(hidden_two, output_size),
        )

    def forward(self, state):
        return self.layers(state)


class DQNAgent:
    def __init__(
        self,
        state_size,
        action_size,
        seed=None,
        hidden_sizes=(128, 64),
        gamma=0.99,
        learning_rate=0.001,
        batch_size=32,
        replay_capacity=10000,
        epsilon_start=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.995,
        target_sync_steps=100,
        device=None,
    ):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = gamma
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.hidden_sizes = tuple(hidden_sizes)
        self.epsilon = epsilon_start
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.target_sync_steps = target_sync_steps
        self.device = torch.device(device or DEVICE)

        self._rng = random.Random(seed)
        self._train_steps = 0

        if seed is not None:
            torch.manual_seed(seed)

        self.q_network = QNetwork(state_size, action_size, self.hidden_sizes).to(self.device)
        self.target_network = QNetwork(state_size, action_size, self.hidden_sizes).to(self.device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()

        self.optimizer = torch.optim.Adam(self.q_network.parameters(), lr=learning_rate)
        self.replay_buffer = ReplayBuffer(replay_capacity)

    def select_action(self, state, available_actions, explore=True):
        if not available_actions:
            raise ValueError('available_actions must not be empty')

        if explore and self._rng.random() < self.epsilon:
            return self._rng.choice(available_actions)

        state_tensor = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
        self.q_network.eval()
        with torch.no_grad():
            q_values = self.q_network(state_tensor).squeeze(0)
        self.q_network.train()

        best_action = available_actions[0]
        best_value = q_values[best_action].item()
        for action in available_actions[1:]:
            value = q_values[action].item()
            if value > best_value:
                best_action = action
                best_value = value
        return best_action

    def remember(self, state, action, reward, next_state, done, next_actions):
        self.replay_buffer.add(
            Experience(
                state=list(state),
                action=action,
                reward=reward,
                next_state=list(next_state),
                done=done,
                next_actions=list(next_actions),
            )
        )

    def train_step(self):
        if len(self.replay_buffer) < self.batch_size:
            return None

        batch = self.replay_buffer.sample(self.batch_size, self._rng)
        states = torch.tensor(
            [experience.state for experience in batch],
            dtype=torch.float32,
            device=self.device,
        )
        actions = torch.tensor(
            [experience.action for experience in batch],
            dtype=torch.long,
            device=self.device,
        )
        rewards = torch.tensor(
            [experience.reward for experience in batch],
            dtype=torch.float32,
            device=self.device,
        )
        dones = torch.tensor(
            [float(experience.done) for experience in batch],
            dtype=torch.float32,
            device=self.device,
        )
        next_states = torch.tensor(
            [experience.next_state for experience in batch],
            dtype=torch.float32,
            device=self.device,
        )

        predicted_q_values = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            next_q_values = self.target_network(next_states)
            masked_next_q = []
            for row, experience in zip(next_q_values, batch):
                if experience.done or not experience.next_actions:
                    masked_next_q.append(0.0)
                    continue
                action_indexes = torch.tensor(experience.next_actions, dtype=torch.long, device=self.device)
                masked_next_q.append(torch.max(row[action_indexes]).item())

            max_next_q_values = torch.tensor(masked_next_q, dtype=torch.float32, device=self.device)
            targets = rewards + (1.0 - dones) * self.gamma * max_next_q_values

        loss = F.mse_loss(predicted_q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=1.0)
        self.optimizer.step()

        self._train_steps += 1
        if self._train_steps % self.target_sync_steps == 0:
            self.sync_target_network()
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        return loss.item()

    def sync_target_network(self):
        self.target_network.load_state_dict(self.q_network.state_dict())


## 4. Visualization and Experiment Helpers

The next cell adds a few helper functions that make the notebook feel more like a live demo:

- a styled board renderer
- a simple SVG chart for training progress
- training and evaluation utilities
- a trajectory viewer for watching one full game


In [ ]:
NUMBER_COLORS = {
    '1': '#2563eb',
    '2': '#16a34a',
    '3': '#dc2626',
    '4': '#7c3aed',
    '5': '#b45309',
    '6': '#0f766e',
    '7': '#111827',
    '8': '#475569',
}


def clone_board(board):
    return [row[:] for row in board]


def render_board_html(board, title=None, subtitle=None, highlight=None):
    size = len(board)
    cells = []
    for row_index, row in enumerate(board):
        for col_index, value in enumerate(row):
            label = '·' if value == ' ' else value
            if value == '?':
                background = '#1d3557'
                color = '#f1faee'
            elif value == ' ':
                background = '#f1faee'
                color = '#94a3b8'
            else:
                background = '#ffffff'
                color = NUMBER_COLORS.get(value, '#111827')

            border = '3px solid #e76f51' if highlight == (row_index, col_index) else '1px solid #cbd5e1'
            cells.append(
                f"<div style='width:54px;height:54px;display:flex;align-items:center;justify-content:center;"
                f"font-size:24px;font-weight:700;border-radius:12px;border:{border};"
                f"background:{background};color:{color};box-shadow:0 8px 20px rgba(15,23,42,0.08);'>"
                f"{escape(str(label))}</div>"
            )

    header = ''
    if title:
        header += f"<div style='font-size:20px;font-weight:800;color:#0f172a;margin-bottom:4px;'>{escape(title)}</div>"
    if subtitle:
        header += f"<div style='font-size:14px;color:#475569;margin-bottom:12px;'>{escape(subtitle)}</div>"

    grid = ''.join(cells)
    return (
        "<div style='display:inline-block;padding:18px 20px;margin:10px 0;background:linear-gradient(180deg,#ffffff 0%,#f8fafc 100%);border:1px solid #e2e8f0;border-radius:18px;'>"
        + header
        + f"<div style='display:grid;grid-template-columns:repeat({size}, 54px);gap:8px;'>{grid}</div>"
        + '</div>'
    )


def show_board(board, title=None, subtitle=None, highlight=None):
    display(HTML(render_board_html(board, title=title, subtitle=subtitle, highlight=highlight)))


def moving_average(values, window=20):
    averaged = []
    for index in range(len(values)):
        start = max(0, index - window + 1)
        averaged.append(sum(values[start:index + 1]) / (index - start + 1))
    return averaged


def line_svg(series, width=720, height=220, color='#0f766e', fill='#ccfbf1', percent=False):
    if not series:
        return '<div>No data yet.</div>'

    values = np.array(series, dtype=float)
    min_value = float(values.min())
    max_value = float(values.max())
    if max_value == min_value:
        max_value = min_value + 1.0

    def scale_x(index):
        if len(values) == 1:
            return 30
        return 30 + index * (width - 60) / (len(values) - 1)

    def scale_y(value):
        return height - 30 - (value - min_value) * (height - 60) / (max_value - min_value)

    points = ' '.join(f'{scale_x(index):.2f},{scale_y(value):.2f}' for index, value in enumerate(values))
    area_points = f'30,{height - 30} ' + points + f' {width - 30},{height - 30}'

    label_low = f'{min_value:.2f}' if not percent else f'{min_value:.0%}'
    label_high = f'{max_value:.2f}' if not percent else f'{max_value:.0%}'

    return f"""
    <svg width='{width}' height='{height}' viewBox='0 0 {width} {height}' style='background:#ffffff;border-radius:16px;border:1px solid #e2e8f0;'>
      <line x1='30' y1='{height - 30}' x2='{width - 30}' y2='{height - 30}' stroke='#cbd5e1' stroke-width='2'/>
      <line x1='30' y1='30' x2='30' y2='{height - 30}' stroke='#cbd5e1' stroke-width='2'/>
      <polygon points='{area_points}' fill='{fill}' opacity='0.7'></polygon>
      <polyline points='{points}' fill='none' stroke='{color}' stroke-width='4' stroke-linecap='round' stroke-linejoin='round'></polyline>
      <text x='36' y='28' font-size='12' fill='#475569'>{escape(label_high)}</text>
      <text x='36' y='{height - 10}' font-size='12' fill='#475569'>{escape(label_low)}</text>
      <text x='{width - 110}' y='{height - 10}' font-size='12' fill='#475569'>episode</text>
    </svg>
    """


def display_training_dashboard(history):
    reward_svg = line_svg(history['smoothed_rewards'], color='#0f766e', fill='#ccfbf1')
    win_svg = line_svg(history['rolling_win_rates'], color='#2563eb', fill='#dbeafe', percent=True)
    loss_svg = line_svg(history['losses'], color='#dc2626', fill='#fee2e2') if history['losses'] else '<div style="padding:24px;border:1px solid #e2e8f0;border-radius:16px;background:#fff;">Loss values appear after the replay buffer fills up.</div>'

    html = f"""
    <div style='display:grid;grid-template-columns:1fr;gap:18px;margin-top:12px;'>
      <div>
        <div style='font-size:18px;font-weight:800;color:#0f172a;margin-bottom:8px;'>Smoothed reward</div>
        {reward_svg}
      </div>
      <div>
        <div style='font-size:18px;font-weight:800;color:#0f172a;margin-bottom:8px;'>Rolling win rate</div>
        {win_svg}
      </div>
      <div>
        <div style='font-size:18px;font-weight:800;color:#0f172a;margin-bottom:8px;'>Training loss</div>
        {loss_svg}
      </div>
    </div>
    """
    display(HTML(html))


def run_episode(env, agent, training, max_steps, capture=False):
    state = env.reset()
    total_reward = 0.0
    won = False
    losses = []
    trajectory = []

    if capture:
        trajectory.append({
            'board': clone_board(env.render()),
            'action': None,
            'reward': 0.0,
            'total_reward': 0.0,
            'note': 'Initial board',
            'highlight': None,
        })

    for _ in range(max_steps):
        available_actions = env.available_actions()
        action = agent.select_action(state, available_actions, explore=training)
        next_state, reward, done, info = env.step(action)
        next_actions = [] if done else env.available_actions()

        if training:
            agent.remember(state, action, reward, next_state, done, next_actions)
            loss = agent.train_step()
            if loss is not None:
                losses.append(loss)

        total_reward += reward
        state = next_state

        if capture:
            row, col = divmod(action, env.size())
            if info.get('hit_mine', False):
                note = 'The agent hit a mine.'
            elif done:
                note = 'The board is solved.'
            else:
                note = f"Revealed {info.get('revealed', 0)} safe cell(s)."
            trajectory.append({
                'board': clone_board(env.render()),
                'action': action,
                'reward': reward,
                'total_reward': total_reward,
                'note': note,
                'highlight': (row, col),
            })

        if done:
            won = not info.get('hit_mine', False)
            break

    average_loss = sum(losses) / len(losses) if losses else None
    return {
        'reward': total_reward,
        'won': won,
        'average_loss': average_loss,
        'trajectory': trajectory,
        'board': clone_board(env.render()),
    }


def evaluate_agent(env, agent, episodes=40, max_steps=None):
    max_steps = max_steps or env.action_space_size() * 2
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0
    wins = 0
    rewards = []
    last_board = None

    for _ in range(episodes):
        outcome = run_episode(env, agent, training=False, max_steps=max_steps, capture=False)
        wins += int(outcome['won'])
        rewards.append(outcome['reward'])
        last_board = outcome['board']

    agent.epsilon = original_epsilon
    return {
        'win_rate': wins / episodes if episodes else 0.0,
        'average_reward': sum(rewards) / len(rewards) if rewards else 0.0,
        'last_board': last_board,
    }


def train_agent(
    size=4,
    num_mines=2,
    episodes=250,
    eval_episodes=30,
    report_every=25,
    safe_first_move=True,
    batch_size=16,
):
    env = MinesweeperEnv(size=size, num_mines=num_mines, safe_first_move=safe_first_move, seed=SEED)
    initial_state = env.reset(seed=SEED)
    agent = DQNAgent(
        state_size=len(initial_state),
        action_size=env.action_space_size(),
        seed=SEED,
        batch_size=batch_size,
        device=DEVICE,
    )
    max_steps = env.action_space_size() * 2

    episode_rewards = []
    rolling_win_rates = []
    losses = []
    recent_rewards = deque(maxlen=report_every)
    recent_wins = deque(maxlen=report_every)

    for episode in range(1, episodes + 1):
        outcome = run_episode(env, agent, training=True, max_steps=max_steps, capture=False)
        episode_rewards.append(outcome['reward'])
        recent_rewards.append(outcome['reward'])
        recent_wins.append(int(outcome['won']))
        rolling_win_rates.append(sum(recent_wins) / len(recent_wins))
        if outcome['average_loss'] is not None:
            losses.append(outcome['average_loss'])

        if episode % report_every == 0:
            print(
                f"Episode {episode:4d} | avg reward {sum(recent_rewards) / len(recent_rewards):6.3f} | "
                f"win rate {sum(recent_wins) / len(recent_wins):5.1%} | epsilon {agent.epsilon:5.3f}"
            )

    history = {
        'episode_rewards': episode_rewards,
        'smoothed_rewards': moving_average(episode_rewards, window=20),
        'rolling_win_rates': rolling_win_rates,
        'losses': losses,
    }
    evaluation = evaluate_agent(env, agent, episodes=eval_episodes, max_steps=max_steps)
    return env, agent, history, evaluation


def show_trajectory(trajectory):
    cards = []
    for step, frame in enumerate(trajectory):
        action_label = 'no action yet' if frame['action'] is None else f"action {frame['action']}"
        subtitle = f"{action_label} | reward {frame['reward']:.2f} | total {frame['total_reward']:.2f}"
        cards.append(
            "<div style='min-width:340px;'>"
            + render_board_html(frame['board'], title=f'Step {step}', subtitle=subtitle + ' | ' + frame['note'], highlight=frame['highlight'])
            + '</div>'
        )
    display(HTML("<div style='display:flex;gap:16px;overflow-x:auto;padding:8px 2px 14px 2px;'>" + ''.join(cards) + '</div>'))


## 5. Quick Visual Sanity Check

Before training, it helps to inspect how the board updates after a few actions.


In [ ]:
demo_env = MinesweeperEnv(size=5, num_mines=5, safe_first_move=True, seed=SEED)
demo_env.reset(seed=SEED)
show_board(demo_env.render(), title='Fresh game', subtitle='All cells start hidden.')

for action in [12, 0, 24]:
    _, reward, done, info = demo_env.step(action)
    row, col = divmod(action, demo_env.size())
    subtitle = f"Clicked cell ({row}, {col}) | reward {reward:.2f} | revealed {info.get('revealed', 0)}"
    show_board(demo_env.render(), title='Board after one action', subtitle=subtitle, highlight=(row, col))
    if done:
        break


## 6. Train the DQN Agent

For a classroom demo, we will train on a small `4x4` board with `2` mines. That setup is simple enough to learn from, but still large enough to show the challenges of exploration.


In [ ]:
env, agent, history, evaluation = train_agent(
    size=4,
    num_mines=2,
    episodes=250,
    eval_episodes=30,
    report_every=25,
    safe_first_move=True,
    batch_size=16,
)

display_training_dashboard(history)
display(Markdown(
    f"**Evaluation win rate:** {evaluation['win_rate']:.1%}  \n"
    f"**Evaluation average reward:** {evaluation['average_reward']:.3f}  \n"
    f"**Final epsilon:** {agent.epsilon:.3f}"
))
show_board(evaluation['last_board'], title='One evaluation board', subtitle='This is a final greedy-policy game after training.')


## 7. Watch the Trained Agent Play

The next cell runs one full greedy episode and shows each move as a separate frame so students can follow the decision sequence.


In [ ]:
rollout = run_episode(env, agent, training=False, max_steps=env.action_space_size() * 2, capture=True)
display(Markdown(
    f"**Result:** {'Win' if rollout['won'] else 'Loss'}  \n"
    f"**Total reward:** {rollout['reward']:.3f}"
))
show_trajectory(rollout['trajectory'])


## 8. Policy Gradient Methods

DQN learns action values and then chooses actions from those values. Policy-gradient methods take a different route: they learn the policy directly.

In the next cells, we compare three policy-gradient approaches:

- `REINFORCE`
- `REINFORCE` with a learned baseline
- `Actor-Critic`


In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, input_size, output_size, hidden_sizes=(128, 64)):
        super().__init__()
        hidden_one, hidden_two = hidden_sizes
        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_one),
            nn.ReLU(),
            nn.Linear(hidden_one, hidden_two),
            nn.ReLU(),
            nn.Linear(hidden_two, output_size),
        )

    def forward(self, state):
        return self.layers(state)


class ValueNetwork(nn.Module):
    def __init__(self, input_size, hidden_sizes=(128, 64)):
        super().__init__()
        hidden_one, hidden_two = hidden_sizes
        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_one),
            nn.ReLU(),
            nn.Linear(hidden_one, hidden_two),
            nn.ReLU(),
            nn.Linear(hidden_two, 1),
        )

    def forward(self, state):
        return self.layers(state).squeeze(-1)


class PolicyGradientAgent:
    def __init__(
        self,
        state_size,
        action_size,
        seed=None,
        hidden_sizes=(128, 64),
        gamma=0.99,
        learning_rate=0.001,
        entropy_coef=0.01,
        device=None,
    ):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = gamma
        self.entropy_coef = entropy_coef
        self.device = torch.device(device or DEVICE)
        self._rng = random.Random(seed)

        if seed is not None:
            torch.manual_seed(seed)

        self.policy_network = PolicyNetwork(
            state_size,
            action_size,
            hidden_sizes=hidden_sizes,
        ).to(self.device)
        self.policy_optimizer = torch.optim.Adam(
            self.policy_network.parameters(),
            lr=learning_rate,
        )

    def start_episode(self):
        pass

    def _masked_distribution(self, state, available_actions):
        if not available_actions:
            raise ValueError('available_actions must not be empty')

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)
        logits = self.policy_network(state_tensor).squeeze(0)
        masked_logits = torch.full_like(logits, -1e9)
        action_indexes = torch.tensor(
            available_actions,
            dtype=torch.long,
            device=self.device,
        )
        masked_logits[action_indexes] = logits[action_indexes]
        distribution = torch.distributions.Categorical(logits=masked_logits)
        return state_tensor, distribution

    def select_action(self, state, available_actions, explore=True, track=True):
        _, distribution = self._masked_distribution(state, available_actions)
        if explore:
            action_tensor = distribution.sample()
        else:
            action_tensor = torch.argmax(distribution.probs)
        action = int(action_tensor.item())
        if track:
            self._record_action(state, distribution, action_tensor)
        return action

    def _record_action(self, state, distribution, action_tensor):
        raise NotImplementedError

    def observe(self, reward, next_state, next_available_actions, done, training=True):
        raise NotImplementedError

    def finish_episode(self, training=True):
        return {}

    def _discounted_returns(self, rewards):
        returns = []
        running_total = 0.0
        for reward in reversed(rewards):
            running_total = reward + self.gamma * running_total
            returns.append(running_total)
        returns.reverse()
        return torch.tensor(returns, dtype=torch.float32, device=self.device)


class REINFORCEAgent(PolicyGradientAgent):
    def start_episode(self):
        self._log_probs = []
        self._rewards = []
        self._entropies = []

    def _record_action(self, state, distribution, action_tensor):
        self._log_probs.append(distribution.log_prob(action_tensor))
        self._entropies.append(distribution.entropy())

    def observe(self, reward, next_state, next_available_actions, done, training=True):
        self._rewards.append(reward)
        return None

    def finish_episode(self, training=True):
        if not training or not self._rewards:
            return {}

        returns = self._discounted_returns(self._rewards)
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std(unbiased=False) + 1e-8)

        log_probs = torch.stack(self._log_probs)
        entropies = torch.stack(self._entropies)
        policy_loss = -(log_probs * returns).sum() - self.entropy_coef * entropies.sum()

        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        nn.utils.clip_grad_norm_(self.policy_network.parameters(), max_norm=1.0)
        self.policy_optimizer.step()

        return {
            'policy_loss': float(policy_loss.item()),
            'value_loss': None,
            'entropy': float(entropies.mean().item()),
        }


class ReinforceBaselineAgent(PolicyGradientAgent):
    def __init__(
        self,
        state_size,
        action_size,
        seed=None,
        hidden_sizes=(128, 64),
        gamma=0.99,
        learning_rate=0.001,
        value_learning_rate=0.001,
        entropy_coef=0.01,
        device=None,
    ):
        super().__init__(
            state_size,
            action_size,
            seed=seed,
            hidden_sizes=hidden_sizes,
            gamma=gamma,
            learning_rate=learning_rate,
            entropy_coef=entropy_coef,
            device=device,
        )
        self.value_network = ValueNetwork(state_size, hidden_sizes=hidden_sizes).to(self.device)
        self.value_optimizer = torch.optim.Adam(
            self.value_network.parameters(),
            lr=value_learning_rate,
        )

    def start_episode(self):
        self._log_probs = []
        self._rewards = []
        self._entropies = []
        self._values = []

    def _record_action(self, state, distribution, action_tensor):
        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)
        self._log_probs.append(distribution.log_prob(action_tensor))
        self._entropies.append(distribution.entropy())
        self._values.append(self.value_network(state_tensor).squeeze(0))

    def observe(self, reward, next_state, next_available_actions, done, training=True):
        self._rewards.append(reward)
        return None

    def finish_episode(self, training=True):
        if not training or not self._rewards:
            return {}

        returns = self._discounted_returns(self._rewards)
        values = torch.stack(self._values)
        advantages = returns - values.detach()
        log_probs = torch.stack(self._log_probs)
        entropies = torch.stack(self._entropies)

        policy_loss = -(log_probs * advantages).sum() - self.entropy_coef * entropies.sum()
        value_loss = F.mse_loss(values, returns)

        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        nn.utils.clip_grad_norm_(self.policy_network.parameters(), max_norm=1.0)
        self.policy_optimizer.step()

        self.value_optimizer.zero_grad()
        value_loss.backward()
        nn.utils.clip_grad_norm_(self.value_network.parameters(), max_norm=1.0)
        self.value_optimizer.step()

        return {
            'policy_loss': float(policy_loss.item()),
            'value_loss': float(value_loss.item()),
            'entropy': float(entropies.mean().item()),
        }


class ActorCriticAgent(PolicyGradientAgent):
    def __init__(
        self,
        state_size,
        action_size,
        seed=None,
        hidden_sizes=(128, 64),
        gamma=0.99,
        learning_rate=0.001,
        value_coef=0.5,
        entropy_coef=0.01,
        device=None,
    ):
        super().__init__(
            state_size,
            action_size,
            seed=seed,
            hidden_sizes=hidden_sizes,
            gamma=gamma,
            learning_rate=learning_rate,
            entropy_coef=entropy_coef,
            device=device,
        )
        self.value_network = ValueNetwork(state_size, hidden_sizes=hidden_sizes).to(self.device)
        self.value_coef = value_coef
        self.value_optimizer = torch.optim.Adam(
            self.value_network.parameters(),
            lr=learning_rate,
        )

    def start_episode(self):
        self._last_log_prob = None
        self._last_entropy = None
        self._last_value = None

    def _record_action(self, state, distribution, action_tensor):
        state_tensor = torch.tensor(
            state,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)
        self._last_log_prob = distribution.log_prob(action_tensor)
        self._last_entropy = distribution.entropy()
        self._last_value = self.value_network(state_tensor).squeeze(0)

    def observe(self, reward, next_state, next_available_actions, done, training=True):
        if not training:
            return None

        reward_tensor = torch.tensor(reward, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            if done:
                next_value = torch.tensor(0.0, dtype=torch.float32, device=self.device)
            else:
                next_state_tensor = torch.tensor(
                    next_state,
                    dtype=torch.float32,
                    device=self.device,
                ).unsqueeze(0)
                next_value = self.value_network(next_state_tensor).squeeze(0)

        target = reward_tensor + self.gamma * next_value
        advantage = target - self._last_value

        policy_loss = -self._last_log_prob * advantage.detach()
        value_loss = advantage.pow(2)
        entropy_bonus = self._last_entropy

        self.policy_optimizer.zero_grad()
        self.value_optimizer.zero_grad()

        total_loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy_bonus
        total_loss.backward()

        nn.utils.clip_grad_norm_(self.policy_network.parameters(), max_norm=1.0)
        nn.utils.clip_grad_norm_(self.value_network.parameters(), max_norm=1.0)

        self.policy_optimizer.step()
        self.value_optimizer.step()

        return {
            'policy_loss': float(policy_loss.item()),
            'value_loss': float(value_loss.item()),
            'entropy': float(entropy_bonus.item()),
        }

    def finish_episode(self, training=True):
        return {}


def run_policy_gradient_episode(env, agent, training, max_steps=None, capture=False):
    max_steps = max_steps or env.action_space_size() * 2
    state = env.reset()
    agent.start_episode()
    total_reward = 0.0
    won = False
    metrics = []
    trajectory = []

    if capture:
        trajectory.append({
            'board': clone_board(env.render()),
            'action': None,
            'reward': 0.0,
            'total_reward': 0.0,
            'note': 'Initial board',
            'highlight': None,
        })

    for _ in range(max_steps):
        available_actions = env.available_actions()
        action = agent.select_action(
            state,
            available_actions,
            explore=training,
            track=training,
        )
        next_state, reward, done, info = env.step(action)
        next_actions = [] if done else env.available_actions()
        metric = agent.observe(
            reward,
            next_state,
            next_actions,
            done,
            training=training,
        )
        if metric:
            metrics.append(metric)

        total_reward += reward
        if capture:
            row, col = divmod(action, env.size())
            if info.get('hit_mine', False):
                note = 'The agent hit a mine.'
            elif done:
                note = 'The board is solved.'
            else:
                note = f"Revealed {info.get('revealed', 0)} safe cell(s)."
            trajectory.append({
                'board': clone_board(env.render()),
                'action': action,
                'reward': reward,
                'total_reward': total_reward,
                'note': note,
                'highlight': (row, col),
            })

        state = next_state
        if done:
            won = not info.get('hit_mine', False)
            break

    final_metric = agent.finish_episode(training=training)
    if final_metric:
        metrics.append(final_metric)

    return {
        'reward': total_reward,
        'won': won,
        'metrics': metrics,
        'trajectory': trajectory,
        'board': clone_board(env.render()),
    }


def train_policy_gradient_agent(env, agent, episodes, report_every=25, max_steps=None):
    max_steps = max_steps or env.action_space_size() * 2
    episode_rewards = []
    rolling_win_rates = []
    policy_losses = []
    value_losses = []
    entropies = []
    recent_rewards = deque(maxlen=report_every)
    recent_wins = deque(maxlen=report_every)

    for episode in range(1, episodes + 1):
        outcome = run_policy_gradient_episode(
            env,
            agent,
            training=True,
            max_steps=max_steps,
            capture=False,
        )
        episode_rewards.append(outcome['reward'])
        recent_rewards.append(outcome['reward'])
        recent_wins.append(int(outcome['won']))
        rolling_win_rates.append(sum(recent_wins) / len(recent_wins))

        for metric in outcome['metrics']:
            if metric.get('policy_loss') is not None:
                policy_losses.append(metric['policy_loss'])
            if metric.get('value_loss') is not None:
                value_losses.append(metric['value_loss'])
            if metric.get('entropy') is not None:
                entropies.append(metric['entropy'])

        if episode % report_every == 0:
            print(
                f"Episode {episode:4d} | avg reward "
                f"{sum(recent_rewards) / len(recent_rewards):6.3f} | "
                f"win rate {sum(recent_wins) / len(recent_wins):5.1%}"
            )

    return {
        'episode_rewards': episode_rewards,
        'smoothed_rewards': moving_average(episode_rewards, window=20),
        'rolling_win_rates': rolling_win_rates,
        'policy_losses': policy_losses,
        'value_losses': value_losses,
        'entropies': entropies,
    }


def evaluate_policy_gradient_agent(env, agent, episodes=30, max_steps=None, capture=False):
    max_steps = max_steps or env.action_space_size() * 2
    wins = 0
    rewards = []
    trajectory = None
    last_board = None

    for episode in range(episodes):
        outcome = run_policy_gradient_episode(
            env,
            agent,
            training=False,
            max_steps=max_steps,
            capture=capture and episode == episodes - 1,
        )
        wins += int(outcome['won'])
        rewards.append(outcome['reward'])
        last_board = outcome['board']
        if capture and episode == episodes - 1:
            trajectory = outcome['trajectory']

    return {
        'win_rate': wins / episodes if episodes else 0.0,
        'average_reward': sum(rewards) / len(rewards) if rewards else 0.0,
        'last_board': last_board,
        'trajectory': trajectory,
    }


In [ ]:
def display_policy_dashboard(history, title):
    reward_svg = line_svg(history['smoothed_rewards'], color='#7c3aed', fill='#ede9fe')
    win_svg = line_svg(history['rolling_win_rates'], color='#2563eb', fill='#dbeafe', percent=True)
    policy_svg = line_svg(history['policy_losses'], color='#dc2626', fill='#fee2e2') if history['policy_losses'] else '<div style="padding:24px;border:1px solid #e2e8f0;border-radius:16px;background:#fff;">No policy-loss values recorded.</div>'
    value_svg = line_svg(history['value_losses'], color='#ea580c', fill='#ffedd5') if history['value_losses'] else '<div style="padding:24px;border:1px solid #e2e8f0;border-radius:16px;background:#fff;">This method does not use a value-loss term.</div>'

    display(HTML(
        f"<div style='margin-top:22px;'>"
        f"<div style='font-size:22px;font-weight:800;color:#0f172a;margin-bottom:14px;'>{escape(title)}</div>"
        f"<div style='display:grid;grid-template-columns:1fr;gap:18px;'>"
        f"<div><div style='font-size:18px;font-weight:700;margin-bottom:8px;'>Smoothed reward</div>{reward_svg}</div>"
        f"<div><div style='font-size:18px;font-weight:700;margin-bottom:8px;'>Rolling win rate</div>{win_svg}</div>"
        f"<div><div style='font-size:18px;font-weight:700;margin-bottom:8px;'>Policy loss</div>{policy_svg}</div>"
        f"<div><div style='font-size:18px;font-weight:700;margin-bottom:8px;'>Value loss</div>{value_svg}</div>"
        f"</div>"
        f"</div>"
    ))


def train_policy_method(agent_class, label, episodes=160, size=4, num_mines=2, **agent_kwargs):
    env = MinesweeperEnv(size=size, num_mines=num_mines, safe_first_move=True, seed=SEED)
    initial_state = env.reset(seed=SEED)
    agent = agent_class(
        state_size=len(initial_state),
        action_size=env.action_space_size(),
        seed=SEED,
        device=DEVICE,
        **agent_kwargs,
    )
    history = train_policy_gradient_agent(
        env,
        agent,
        episodes=episodes,
        report_every=20,
        max_steps=env.action_space_size() * 2,
    )
    evaluation = evaluate_policy_gradient_agent(
        env,
        agent,
        episodes=20,
        max_steps=env.action_space_size() * 2,
        capture=True,
    )
    return {
        'label': label,
        'env': env,
        'agent': agent,
        'history': history,
        'evaluation': evaluation,
    }


def display_policy_summary(results):
    cards = []
    for result in results:
        evaluation = result['evaluation']
        cards.append(
            f"<div style='padding:18px 20px;border:1px solid #e2e8f0;border-radius:16px;background:#fff;min-width:220px;'>"
            f"<div style='font-size:18px;font-weight:800;color:#0f172a;margin-bottom:10px;'>{escape(result['label'])}</div>"
            f"<div style='font-size:14px;color:#475569;'>Win rate</div>"
            f"<div style='font-size:28px;font-weight:800;color:#2563eb;margin-bottom:10px;'>{evaluation['win_rate']:.1%}</div>"
            f"<div style='font-size:14px;color:#475569;'>Average reward</div>"
            f"<div style='font-size:24px;font-weight:800;color:#0f766e;'>{evaluation['average_reward']:.3f}</div>"
            f"</div>"
        )
    display(HTML("<div style='display:flex;gap:14px;flex-wrap:wrap;margin:16px 0;'>" + ''.join(cards) + '</div>'))


## 9. Train and Compare Policy-Gradient Agents

This comparison uses the same small classroom setting as the DQN demo: a `4x4` board with `2` mines and a safe first move.


In [ ]:
policy_results = []

for label, agent_class in [
    ('REINFORCE', REINFORCEAgent),
    ('REINFORCE with baseline', ReinforceBaselineAgent),
    ('Actor-Critic', ActorCriticAgent),
]:
    print(f'Training {label}...')
    policy_results.append(train_policy_method(agent_class, label, episodes=160, size=4, num_mines=2))

display_policy_summary(policy_results)

for result in policy_results:
    display_policy_dashboard(result['history'], result['label'])
    show_board(
        result['evaluation']['last_board'],
        title=result['label'],
        subtitle=f"Evaluation win rate {result['evaluation']['win_rate']:.1%} | average reward {result['evaluation']['average_reward']:.3f}",
    )


## 10. Watch the Best Policy-Gradient Agent

We now select the best policy-gradient method from the evaluation run and inspect one full greedy rollout.


In [ ]:
best_policy_result = max(policy_results, key=lambda result: result['evaluation']['win_rate'])
display(Markdown(
    f"**Best policy-gradient method:** {best_policy_result['label']}  \n"
    f"**Win rate:** {best_policy_result['evaluation']['win_rate']:.1%}  \n"
    f"**Average reward:** {best_policy_result['evaluation']['average_reward']:.3f}"
))
show_trajectory(best_policy_result['evaluation']['trajectory'])


## Discussion

- Why is Minesweeper hard for RL?
- Which parts of the reward function encourage useful behavior?
- What happens if we increase the number of mines?
- What happens if we increase the size of the board?
- How might we redesign the state representation to include more structure?
- Would a convolutional network be a better fit than a plain multilayer perceptron?
